# Week 3: Healthcare Operations, Statistical Investigation & Insight Discovery

**AnalystLab Africa · Data Science Internship** · **Author:** Molemi Molokoane

**Dataset:** healthcare dataset pack supplied by AnalystLab Africa (encounters, procedures, patients, payers, organizations, data dictionary). The records cover one hospital, Massachusetts General Hospital, from January 2011 to February 2022. Patient names in the pack end in generated digit sequences, and the field descriptions match the documentation of [Synthea](https://github.com/synthetichealth/synthea), MITRE's open-source synthetic patient generator, so the data appears to be **synthetic**. It is still treated as if it were real patient data.

**Research question:** *How do recorded total claim costs vary across encounter classes, and is encounter duration associated with cost?*

### Workflow
1. Business understanding and success criteria
2. Data inspection
3. Data preparation
4. Exploratory data analysis
5. Hypothesis 1: claim cost differs across encounter classes
6. Hypothesis 2: longer encounters cost more within a class
7. Validation
8. Limitations
9. Findings and recommendations

## 0. Setup

Runs in Google Colab or locally. The raw data pack is **not** stored in the public GitHub repository because `patients.csv` contains names and addresses, so in a fresh Colab session the setup cell asks you to upload the six CSV files.

In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter
from scipy import stats

# Display and plotting defaults
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

RANDOM_STATE = 42  # fixed seed so the bootstrap intervals are identical on every run

# Paths: raw files in Data/, charts in Visualizations/, aggregate result tables in Outputs/
DATA_DIR, VIZ_DIR, OUT_DIR = "Data", "Visualizations", "Outputs"
for folder in (DATA_DIR, VIZ_DIR, OUT_DIR):
    os.makedirs(folder, exist_ok=True)

# Encounter classes in a fixed order (largest to smallest), used by every table and chart
CLASS_ORDER = ["ambulatory", "outpatient", "urgentcare", "emergency", "wellness", "inpatient"]

# Colours: one colour for single-series charts, a fixed categorical order for multi-series charts
BAR_COLOR = "#2a78d6"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK = "#0b0b0b"
dollars = FuncFormatter(lambda v, _: f"${v:,.0f}")

def save_fig(fig, name):
    # Save a chart to Visualizations/ and display it
    fig.savefig(os.path.join(VIZ_DIR, name), bbox_inches="tight", dpi=150)
    plt.show()

In [ ]:
# Make sure the data pack is available; in Colab, upload it if it is not
REQUIRED_FILES = ["encounters.csv", "procedures.csv", "patients.csv", "payers.csv",
                  "organizations.csv", "data_dictionary.csv"]
missing_files = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing_files:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Copy these files into {DATA_DIR}/ and rerun: {missing_files}")
    print("Select these files in the upload dialog:", missing_files)
    for name, content in files.upload().items():
        with open(os.path.join(DATA_DIR, name), "wb") as fh:
            fh.write(content)
    missing_files = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(DATA_DIR, f))]
    assert not missing_files, f"Still missing: {missing_files}"

print("All data files present.")

## 1. Business Understanding

**The challenge.** The hospital wants to understand where its encounter costs come from. Claim costs are recorded for every encounter, but nobody has established how much they differ between types of care (ambulatory, emergency, inpatient and so on), or whether longer encounters are reliably more expensive. Without that, budgets and cost reviews rely on averages that may be dominated by a handful of very expensive encounters.

- **Problem investigated:** how recorded total claim costs vary across encounter classes, and whether encounter duration is associated with claim cost within a class.
- **Why it matters:** encounter class is known as soon as a patient arrives, and duration is something operations teams actively manage (length of stay, clinic slot length). If cost differs strongly by class, budgets and cost reviews should be set per class. If duration tracks cost, managing time is also a lever on cost; if it does not, cost drivers lie elsewhere (procedures, medications).
- **Intended user:** hospital operations and finance managers (cost review, budgeting, capacity planning). They are not statisticians, so every result is also stated in plain language.
- **Decisions supported:** (1) whether cost targets and reviews should be set per encounter class rather than hospital-wide; (2) whether encounter duration is a useful proxy for cost when planning capacity and budgets.
- **Limitations of the evidence:**
  - One hospital only, so results cannot be compared across organisations or generalised.
  - The data appears to be synthetic, so patterns reflect the generator's rules as well as real-world behaviour.
  - `TOTAL_CLAIM_COST` is a **recorded claim amount**, not the hospital's actual cost of delivering care.
  - Medication and immunisation line items are included in the total but are not supplied as separate tables, so part of the cost cannot be explained.
  - There is no clinical severity information, so differences between classes cannot be attributed to how sick patients were.
  - Many encounters come from a small number of patients, so encounters are not independent observations.

**Variables required**

| Variable | Source | Role |
|---|---|---|
| `TOTAL_CLAIM_COST` | encounters | outcome |
| `ENCOUNTERCLASS` | encounters | grouping variable (H1) |
| `START`, `STOP` → duration in hours | encounters | explanatory variable (H2) |
| `PATIENT` | encounters | cluster identifier for repeated records |
| `BASE_ENCOUNTER_COST`, procedure `BASE_COST` | encounters, procedures | explain what makes up the total |
| `PAYER` → payer group | encounters, payers | potential confounder |
| `BIRTHDATE` → age at encounter | patients | potential confounder |

### Success criteria
The work is successful if:
1. **Independent reconciliation:** encounter counts, median claim cost per class and the number of encounters over 24 hours, recalculated with a separate script that does not use pandas, match the notebook exactly.
2. **Repeated records handled:** every hypothesis test is run at the encounter level *and* with one value per patient (or a patient-level bootstrap), and a conclusion is only stated when both agree.
3. **Effect sizes, not just p-values:** every hypothesis reports an effect size with a 95% confidence interval from a bootstrap that resamples patients.
4. **Privacy:** no names, addresses or other identifying patient fields appear in any printed output, chart or exported file (checked automatically at the end of the notebook).
5. **Reproducibility:** the notebook runs top to bottom in a fresh Colab session with identical results, and the README explains how to reproduce it.

## 2. Data Inspection

### 2.1 Load the files
`utf-8-sig` strips the byte-order mark that prefixes the first column name in every file.

In [ ]:
def read(name):
    return pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"), encoding="utf-8-sig")

encounters_raw = read("encounters")
procedures_raw = read("procedures")
patients_raw = read("patients")
payers = read("payers")
organizations = read("organizations")
dictionary = read("data_dictionary")

tables = {"encounters": encounters_raw, "procedures": procedures_raw, "patients": patients_raw,
          "payers": payers, "organizations": organizations}
shape_summary = pd.DataFrame({name: {"rows": df.shape[0], "columns": df.shape[1]}
                              for name, df in tables.items()}).T
shape_summary

### 2.2 Structure and data types
Patient rows are never displayed, only the column types.

In [ ]:
encounters_raw.head()

In [ ]:
for name, df in tables.items():
    print(f"--- {name}")
    print(df.dtypes.to_string(), "\n")

Dates (`START`, `STOP`, `BIRTHDATE`, `DEATHDATE`) are read as text and must be converted. The cost columns are already numeric. The IDs are UUID strings.

### 2.3 Missing values

In [ ]:
missing_rows = []
for name, df in tables.items():
    for col in df.columns:
        n = df[col].isna().sum()
        if n:
            missing_rows.append({"table": name, "column": col, "missing": n,
                                 "pct_missing": round(100 * n / len(df), 1)})
missing_summary = pd.DataFrame(missing_rows)
missing_summary

None of the missing values affect the research question. Each one has a structural meaning, so **nothing is imputed**:
- `REASONCODE` / `REASONDESCRIPTION` (70% of encounters, 78% of procedures) are only filled in when the encounter targeted a specific condition, according to the data dictionary.
- `DEATHDATE` is blank for patients who are alive. `SUFFIX` and `MAIDEN` only apply to some people. `MARITAL` and `ZIP` are not used.
- The `NO_INSURANCE` payer has no address because it is not an organisation.

Every variable the analysis uses (class, costs, start/stop, patient, payer, birth date) is complete.

### 2.4 Data dictionary vs actual columns
The brief warns that not every documented field is present. Field names are compared after upper-casing and replacing spaces with underscores.

In [ ]:
dictionary_fields = (dictionary.dropna(subset=["Field"])
                     .assign(field=lambda d: d["Field"].str.upper().str.replace(" ", "_")))
dict_check = []
for name, df in tables.items():
    documented = set(dictionary_fields.loc[dictionary_fields["Table"] == name, "field"])
    actual = {c.upper() for c in df.columns}
    dict_check.append({"table": name,
                       "documented but absent": sorted(documented - actual),
                       "present but undocumented": sorted(actual - documented)})
pd.DataFrame(dict_check)

`patients.csv` is missing `MIDDLE` and `FIPS County Code`. Neither is needed here. All encounter and procedure fields match the dictionary.

### 2.5 Privacy: drop identifying patient fields
`patients.csv` contains names, addresses, birthplace and coordinates. The analysis only needs the patient key, birth date (to derive age), death date and gender. Everything else is dropped now and never printed, charted or exported.

In [ ]:
PATIENT_KEEP = ["Id", "BIRTHDATE", "DEATHDATE", "GENDER"]
print("Dropped identifying columns:", [c for c in patients_raw.columns if c not in PATIENT_KEEP])
patients = patients_raw[PATIENT_KEEP].copy()
del patients_raw
tables["patients"] = patients

### 2.6 Duplicates
Exact duplicate rows and duplicate primary keys are checked first. Then two near-duplicate patterns are checked: encounters for the same patient starting at the same moment, and the same procedure code recorded twice within one encounter.

In [ ]:
duplicate_summary = pd.DataFrame({
    "exact duplicate rows": {n: df.duplicated().sum() for n, df in tables.items()},
    "duplicate Ids": {n: df["Id"].duplicated().sum() if "Id" in df else np.nan for n, df in tables.items()},
})
display(duplicate_summary)

same_start = encounters_raw.duplicated(["PATIENT", "START"], keep=False)
same_start_class = encounters_raw.duplicated(["PATIENT", "ENCOUNTERCLASS", "START"], keep=False)
same_start_full = encounters_raw.duplicated(
    ["PATIENT", "START", "STOP", "ENCOUNTERCLASS", "CODE", "TOTAL_CLAIM_COST"], keep=False)
repeated_procedures = procedures_raw.duplicated(["ENCOUNTER", "CODE", "START"], keep=False)

print(f"Encounters sharing patient + start time:              {same_start.sum():,}")
print(f"  ... and the same class:                              {same_start_class.sum():,}")
print(f"  ... and identical stop, code and claim cost:         {same_start_full.sum():,}")
print(f"Procedures repeated within an encounter (same code+start): {repeated_procedures.sum():,}")

In [ ]:
# What do the same-start encounters look like? (no patient identifiers shown)
encounters_raw[same_start].groupby(["ENCOUNTERCLASS", "DESCRIPTION"]).size().sort_values(ascending=False).head(10)

There are no exact duplicates and no repeated primary keys. Encounters that share a patient and start time all have distinct IDs. The tables above show whether they are genuinely different encounters or likely double entries; the decision is recorded in Section 3.

### 2.7 Relationships between tables
`encounters` is the central table: `PATIENT` → `patients.Id`, `PAYER` → `payers.Id`, `ORGANIZATION` → `organizations.Id`, and `procedures.ENCOUNTER` → `encounters.Id` (one encounter, zero or more procedures).

In [ ]:
key_checks = [
    ("encounters.PATIENT → patients.Id", encounters_raw["PATIENT"], patients["Id"]),
    ("encounters.PAYER → payers.Id", encounters_raw["PAYER"], payers["Id"]),
    ("encounters.ORGANIZATION → organizations.Id", encounters_raw["ORGANIZATION"], organizations["Id"]),
    ("procedures.ENCOUNTER → encounters.Id", procedures_raw["ENCOUNTER"], encounters_raw["Id"]),
    ("procedures.PATIENT → patients.Id", procedures_raw["PATIENT"], patients["Id"]),
]
pd.DataFrame([{"relationship": label, "rows": len(fk), "unmatched": (~fk.isin(pk)).sum()}
              for label, fk, pk in key_checks])

In [ ]:
print("Organisations in the data:", organizations["NAME"].tolist())
print("Encounters with at least one procedure:",
      f"{encounters_raw['Id'].isin(procedures_raw['ENCOUNTER']).sum():,} of {len(encounters_raw):,}")

All foreign keys match: there are no orphan records. Because there is **only one organisation**, hospital-level comparisons are impossible. Only about half of encounters have a procedure, so procedure cost can only partly explain claim cost.

## 3. Data Preparation

No rows are deleted. Problem records are **flagged**, and each analysis states which records it uses, so every exclusion is visible and reversible.

### 3.1 Dates and duration

In [ ]:
enc = encounters_raw.copy()
ISO_UTC = "%Y-%m-%dT%H:%M:%SZ"
for col in ["START", "STOP"]:
    enc[col] = pd.to_datetime(enc[col], format=ISO_UTC, utc=True, errors="coerce")

enc["duration_h"] = (enc["STOP"] - enc["START"]).dt.total_seconds() / 3600

print("Unparseable START / STOP:", enc["START"].isna().sum(), "/", enc["STOP"].isna().sum())
print("STOP before START:       ", (enc["duration_h"] < 0).sum())
print("Zero-length encounters:  ", (enc["duration_h"] == 0).sum())
print("Date range:              ", enc["START"].min().date(), "to", enc["START"].max().date())
print("Encounters over 24 hours:", (enc["duration_h"] > 24).sum())

In [ ]:
enc["year"] = enc["START"].dt.year
enc["quarter"] = enc["START"].dt.to_period("Q").astype(str)
enc.groupby(enc["START"].dt.strftime("%Y-%m")).size().tail(6)

All timestamps parse and none run backwards. The data stops on **5 February 2022** (one encounter in February), so **2022 is a partial year** and must not be compared with full years.

### 3.2 Implausible durations

In [ ]:
long_enc = enc[enc["duration_h"] > 24]
print(long_enc["ENCOUNTERCLASS"].value_counts().to_string(), "\n")
(long_enc[long_enc["ENCOUNTERCLASS"] != "inpatient"]
 .assign(start_date=lambda d: d["START"].dt.date, duration_days=lambda d: d["duration_h"] / 24)
 [["ENCOUNTERCLASS", "DESCRIPTION", "start_date", "duration_days", "TOTAL_CLAIM_COST"]]
 .sort_values("duration_days", ascending=False))

Inpatient stays longer than a day are expected. The **non-inpatient** encounters above are not: they include follow-up visits and outpatient "encounters for problem" lasting weeks or even years, almost all billed at the minimum $86. These are recording errors (most likely a wrong `STOP` date).

**Decision:** flag them as `duration_suspect`. They stay in the cost analysis (H1), because their cost and class are plausible, but are excluded from the duration analysis (H2).

In [ ]:
enc["duration_suspect"] = (enc["duration_h"] > 24) & (enc["ENCOUNTERCLASS"] != "inpatient")
print("Flagged duration_suspect:", enc["duration_suspect"].sum())
print("Inpatient stays over 30 days:", ((enc["ENCOUNTERCLASS"] == "inpatient") & (enc["duration_h"] > 24 * 30)).sum())

### 3.3 How much does duration actually vary within each class?

In [ ]:
def share_at_mode(s):
    return s.round(3).value_counts(normalize=True).iloc[0]

valid = enc[~enc["duration_suspect"]]
duration_profile = (valid.groupby("ENCOUNTERCLASS")["duration_h"]
                    .agg(distinct_values=lambda s: s.round(3).nunique(), median="median",
                         p95=lambda s: s.quantile(0.95), share_at_most_common_value=share_at_mode)
                    .reindex(CLASS_ORDER))
duration_profile

**Wellness and urgent care encounters all last exactly 15 minutes**, and most emergency and outpatient encounters sit on a single value too. Durations appear to come from fixed templates rather than real recorded times. A correlation needs variation, so H2 can only be tested in classes with more than one distinct duration, and the heavy ties must be kept in mind when reading the results.

In [ ]:
VARIABLE_DURATION_CLASSES = [c for c in CLASS_ORDER if duration_profile.loc[c, "distinct_values"] > 1]
print("Classes usable for the duration analysis:", VARIABLE_DURATION_CLASSES)

### 3.4 What makes up the total claim cost?
Procedures are summed per encounter and joined to encounters. An encounter with no procedure rows genuinely had **zero** procedures, so filling those gaps with 0 is correct here (it is not imputation).

In [ ]:
proc_per_enc = procedures_raw.groupby("ENCOUNTER").agg(n_procedures=("CODE", "size"),
                                                       procedure_cost=("BASE_COST", "sum"))
enc = enc.merge(proc_per_enc, left_on="Id", right_index=True, how="left")
enc[["n_procedures", "procedure_cost"]] = enc[["n_procedures", "procedure_cost"]].fillna(0)
enc["has_procedure"] = enc["n_procedures"] > 0

base_plus_proc = np.isclose(enc["TOTAL_CLAIM_COST"], enc["BASE_ENCOUNTER_COST"] + enc["procedure_cost"], atol=0.01)
base_only = np.isclose(enc["TOTAL_CLAIM_COST"], enc["BASE_ENCOUNTER_COST"], atol=0.01)
print(f"Total = base cost only:                {base_only.mean():.1%}")
print(f"Total = base + procedure costs:        {base_plus_proc.mean():.1%}")
print(f"Total below base cost:                 {(enc['TOTAL_CLAIM_COST'] < enc['BASE_ENCOUNTER_COST'] - 0.01).sum()}")
print("Distinct base encounter costs:", sorted(enc["BASE_ENCOUNTER_COST"].unique()))

Base encounter cost takes only a handful of fixed values ($85–$146), so it carries almost no information. Total claim cost equals base plus procedure cost for only about 30% of encounters. The remainder includes line items, such as medications and immunisations, that are **not supplied** in this pack. That is a limitation for explaining *why* costs differ.

### 3.5 Zero-cost claims

In [ ]:
zero_claim = enc["TOTAL_CLAIM_COST"] == 0
enc[zero_claim].groupby(["ENCOUNTERCLASS", "DESCRIPTION"]).size().sort_values(ascending=False)

The $0 claims are dominated by **death certification** (wellness) and similar administrative encounters. Recording them as free is plausible, so they are kept. Charts on a log scale cannot show $0 and leave them out; this is noted on each chart.

### 3.6 Payer group and patient age (potential confounders)

In [ ]:
enc["payer_name"] = enc["PAYER"].map(payers.set_index("Id")["NAME"])
PAYER_GROUPS = ["Medicare", "Medicaid / Dual", "Private", "No insurance"]
enc["payer_group"] = np.select(
    [enc["payer_name"].eq("Medicare"), enc["payer_name"].isin(["Medicaid", "Dual Eligible"]),
     enc["payer_name"].eq("NO_INSURANCE")],
    ["Medicare", "Medicaid / Dual", "No insurance"], default="Private")

birth = pd.to_datetime(patients.set_index("Id")["BIRTHDATE"], format="%Y-%m-%d", utc=True)
enc["age_at_encounter"] = (enc["START"] - enc["PATIENT"].map(birth)).dt.days / 365.25

print(enc["payer_group"].value_counts().to_string())
print("\nAge at encounter: min {:.1f}, median {:.1f}, max {:.1f}".format(
    enc["age_at_encounter"].min(), enc["age_at_encounter"].median(), enc["age_at_encounter"].max()))

### 3.7 Same-start encounters

In [ ]:
# Decision recorded after reviewing Section 2.6: same-start encounters have distinct IDs and are
# kept as separate encounters; fully identical ones (same stop, code and cost) are flagged.
enc["possible_double_entry"] = same_start_full.values
print("Flagged possible_double_entry:", enc["possible_double_entry"].sum())

### 3.8 Data-quality summary

In [ ]:
data_quality_summary = pd.DataFrame([
    ("Missing REASONCODE / REASONDESCRIPTION", f"{encounters_raw['REASONCODE'].isna().sum():,} encounters",
     "Structural (only set when a condition was targeted); not used, not imputed"),
    ("Dictionary fields absent from CSV", "patients: MIDDLE, FIPS County Code", "Not needed; documented"),
    ("Identifying patient fields", f"{len(PATIENT_KEEP)} of 20 patient columns kept",
     "Names, addresses, coordinates dropped on load"),
    ("Exact duplicates / duplicate IDs", "0 in every table", "None to remove"),
    ("Same patient + start time", f"{same_start.sum():,} encounters",
     "Distinct IDs, kept; fully identical ones flagged possible_double_entry"),
    ("Non-inpatient encounters over 24 h", f"{enc['duration_suspect'].sum()} encounters",
     "Flagged duration_suspect; kept for cost, excluded from duration analysis"),
    ("Template durations", "wellness and urgent care always 15 min",
     "Those classes excluded from the duration analysis"),
    ("Partial year", "2022 covers 1 Jan to 5 Feb only", "Excluded from year-on-year comparisons"),
    ("Zero total claim cost", f"{zero_claim.sum()} encounters", "Kept (mostly death certification)"),
    ("Total claim not explained by supplied tables", f"{1 - base_plus_proc.mean():.0%} of encounters",
     "Medication/immunisation line items not supplied; limitation"),
    ("Single organisation", "1 hospital", "No between-hospital comparison possible"),
    ("Repeated records", f"{enc['PATIENT'].nunique():,} patients, {len(enc):,} encounters",
     "Tests use patient-level values and patient bootstrap"),
], columns=["issue", "extent", "decision"])
data_quality_summary.to_csv(os.path.join(OUT_DIR, "data_quality_summary.csv"), index=False)
data_quality_summary

## 4. Exploratory Data Analysis

### 4.1 Encounter volume by class

In [ ]:
counts = enc["ENCOUNTERCLASS"].value_counts().reindex(CLASS_ORDER)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(CLASS_ORDER[::-1], counts.values[::-1], color=BAR_COLOR, height=0.6)
for y, n in enumerate(counts.values[::-1]):
    ax.text(n + counts.max() * 0.01, y, f"{n:,}  ({n / len(enc):.0%})", va="center", fontsize=9, color=INK)
ax.set(xlabel="Encounters", xlim=(0, counts.max() * 1.22),
       title="Encounters by class, January 2011 to February 2022")
ax.grid(axis="y", visible=False)
save_fig(fig, "01_encounters_by_class.png")

### 4.2 Distribution of total claim cost

In [ ]:
cost = enc["TOTAL_CLAIM_COST"]
positive = cost[cost > 0]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(positive, bins=np.logspace(np.log10(positive.min()), np.log10(positive.max()), 60), color=BAR_COLOR)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(dollars)
for value, label, style in [(cost.median(), "median", "-"), (cost.mean(), "mean", "--")]:
    ax.axvline(value, color=INK, ls=style, lw=1.2)
    ax.text(value, ax.get_ylim()[1] * 0.97, f" {label} ${value:,.0f}", va="top", fontsize=9, color=INK)
ax.set(xlabel="Total claim cost (log scale)", ylabel="Encounters",
       title=f"Total claim cost is heavily right-skewed ({zero_claim.sum()} $0 claims not shown)")
save_fig(fig, "02_claim_cost_distribution.png")

print(cost.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).to_string())
print(f"Skewness: {cost.skew():.1f}")

### 4.3 Claim cost by encounter class

In [ ]:
class_summary = (enc.groupby("ENCOUNTERCLASS")
                 .agg(encounters=("Id", "size"), patients=("PATIENT", "nunique"),
                      median_cost=("TOTAL_CLAIM_COST", "median"),
                      q1=("TOTAL_CLAIM_COST", lambda s: s.quantile(0.25)),
                      q3=("TOTAL_CLAIM_COST", lambda s: s.quantile(0.75)),
                      mean_cost=("TOTAL_CLAIM_COST", "mean"),
                      p95=("TOTAL_CLAIM_COST", lambda s: s.quantile(0.95)),
                      max_cost=("TOTAL_CLAIM_COST", "max"),
                      total_cost=("TOTAL_CLAIM_COST", "sum"),
                      pct_with_procedure=("has_procedure", "mean"),
                      median_age=("age_at_encounter", "median"))
                 .reindex(CLASS_ORDER))
class_summary["share_of_total_cost"] = class_summary["total_cost"] / class_summary["total_cost"].sum()
class_summary.to_csv(os.path.join(OUT_DIR, "class_cost_summary.csv"))
class_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=enc[enc["TOTAL_CLAIM_COST"] > 0], x="TOTAL_CLAIM_COST", y="ENCOUNTERCLASS",
            order=CLASS_ORDER, color=BAR_COLOR, log_scale=True, fliersize=1.5, width=0.6, ax=ax)
ax.scatter(class_summary["mean_cost"], range(len(CLASS_ORDER)), marker="D", s=30, color=INK, zorder=3, label="Mean")
ax.xaxis.set_major_formatter(dollars)
ax.legend(loc="lower right")
ax.set(xlabel="Total claim cost (log scale)", ylabel="",
       title="Claim cost by encounter class: box = middle 50%, line = median ($0 claims not shown)")
save_fig(fig, "03_claim_cost_by_class.png")

### 4.4 Repeated records: encounters per patient

In [ ]:
per_patient = enc.groupby("PATIENT").size()
top10_share = per_patient.nlargest(10).sum() / len(enc)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(per_patient, bins=np.logspace(0, np.log10(per_patient.max()), 40), color=BAR_COLOR)
ax.set_xscale("log")
ax.set(xlabel="Encounters per patient (log scale)", ylabel="Patients",
       title=f"Encounters per patient: median {per_patient.median():.0f}, maximum {per_patient.max():,}\n"
             f"The 10 heaviest users account for {top10_share:.0%} of all encounters")
save_fig(fig, "04_encounters_per_patient.png")
print(per_patient.describe().to_string())

### 4.5 Duration by class

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=valid, x="duration_h", y="ENCOUNTERCLASS", order=CLASS_ORDER, color=BAR_COLOR,
            log_scale=True, fliersize=1.5, width=0.6, ax=ax)
ax.set(xlabel="Duration in hours (log scale)", ylabel="",
       title=f"Encounter duration by class ({enc['duration_suspect'].sum()} implausible durations excluded)")
save_fig(fig, "05_duration_by_class.png")

### 4.6 Duration vs claim cost

In [ ]:
dur_df = enc[~enc["duration_suspect"] & enc["ENCOUNTERCLASS"].isin(VARIABLE_DURATION_CLASSES)]
n_panels = len(VARIABLE_DURATION_CLASSES)
fig, axes = plt.subplots(int(np.ceil(n_panels / 2)), 2, figsize=(10, 3.6 * np.ceil(n_panels / 2)), squeeze=False)
for ax, cls in zip(axes.flat, VARIABLE_DURATION_CLASSES):
    sub = dur_df[(dur_df["ENCOUNTERCLASS"] == cls) & (dur_df["TOTAL_CLAIM_COST"] > 0)]
    ax.scatter(sub["duration_h"], sub["TOTAL_CLAIM_COST"], s=8, alpha=0.25, color=BAR_COLOR, edgecolors="none")
    ax.set(xscale="log", yscale="log", title=f"{cls} (n = {len(sub):,})",
           xlabel="Duration in hours (log)", ylabel="Total claim cost (log)")
    ax.yaxis.set_major_formatter(dollars)
for ax in list(axes.flat)[n_panels:]:
    ax.set_visible(False)
fig.suptitle("Duration vs claim cost, classes with variable duration", y=1.01)
fig.tight_layout()
save_fig(fig, "06_duration_vs_cost.png")

### 4.7 Where the cost comes from: procedures

In [ ]:
with_proc = enc.groupby(["ENCOUNTERCLASS", "has_procedure"])["TOTAL_CLAIM_COST"].median().unstack().reindex(CLASS_ORDER)
x = np.arange(len(CLASS_ORDER))
width = 0.38
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(x - width / 2, with_proc[False], width, color=SERIES[0], label="No procedure recorded")
ax.bar(x + width / 2, with_proc[True], width, color=SERIES[1], label="At least one procedure")
ax.set_xticks(x, [f"{c}\n{class_summary.loc[c, 'pct_with_procedure']:.0%} with procedure" for c in CLASS_ORDER])
ax.set_yscale("log")
ax.yaxis.set_major_formatter(dollars)
ax.legend(frameon=False)
ax.set(ylabel="Median total claim cost (log scale)",
       title="Encounters with a procedure cost far more, in every class")
ax.grid(axis="x", visible=False)
save_fig(fig, "07_cost_with_and_without_procedures.png")
with_proc

### 4.8 Payer mix by class (potential confounder)

In [ ]:
mix = pd.crosstab(enc["ENCOUNTERCLASS"], enc["payer_group"], normalize="index").reindex(CLASS_ORDER)[PAYER_GROUPS]
fig, ax = plt.subplots(figsize=(9, 4.5))
left = np.zeros(len(mix))
for color, group in zip(SERIES, PAYER_GROUPS):
    ax.barh(mix.index, mix[group], left=left, color=color, label=group, edgecolor="white", linewidth=1.5, height=0.65)
    for y, (start, share) in enumerate(zip(left, mix[group])):
        if share >= 0.08:
            ax.text(start + share / 2, y, f"{share:.0%}", ha="center", va="center", fontsize=8, color="white")
    left += mix[group].values
ax.invert_yaxis()
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
ax.set(xlim=(0, 1), title="Payer mix differs by encounter class")
ax.grid(visible=False)
save_fig(fig, "08_payer_mix_by_class.png")

### 4.9 Encounter volume over time

In [ ]:
yearly = enc.groupby("year").size()
full_years = yearly[yearly.index < 2022]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(full_years.index, full_years.values, color=BAR_COLOR, lw=2, marker="o", ms=5, label="Full year")
ax.plot([2022], [yearly[2022]], ls="none", marker="o", ms=8, mfc="white", mec=BAR_COLOR, mew=2,
        label="2022 (1 Jan to 5 Feb only)")
ax.set(xlabel="Year", ylabel="Encounters", ylim=(0, yearly.max() * 1.1), title="Encounters per year")
ax.set_xticks(yearly.index)
ax.legend(frameon=False)
save_fig(fig, "09_encounters_per_year.png")